# SIRA-Tiny and SIRA-Small Paper Comparison on Colab L4

This notebook runs the two smaller attack-model configurations reported by the SIRA paper:

- SIRA-Tiny: Llama 3.2 3B Instruct, bf16
- SIRA-Small: Llama 3 8B Instruct, bf16
- Cognitive Integrity Grid Masking: proposed deterministic 2x2 grid-guided baseline
- Normal Rewrite Control: same rewrite model without grid masking

The paper describes these as Llama3 Instruct 3B and 8B. The released code identifies the intended checkpoints as Llama 3.2 3B Instruct and Meta-Llama-3-8B-Instruct. The grid baseline verifies an external token-state trace; it does not expose or verify hidden chain-of-thought.

Before running, choose **Runtime > Change runtime type > L4 GPU**. Add a Colab Secret named `HF_TOKEN` with access to both gated Llama checkpoints. Inaccessible models are recorded as skipped rather than crashing the run.


In [ ]:
# Check that Colab assigned the requested L4 GPU.
import subprocess

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()

print("GPU:", gpu_name)
if "L4" not in gpu_name:
    raise RuntimeError("This notebook requires an L4 GPU. Change the Colab runtime type and try again.")


In [ ]:
# Experiment settings. Start with 3 samples, then increase toward the paper's 500 samples.
REPO_URL = "https://github.com/hanifnoerr/Self-information-Rewrite-Attack.git"
BRANCH = "codex/browser-colab-l4"
REPO_DIR = "/content/Self-information-Rewrite-Attack"
OUTPUT_ROOT = "/content/sira_outputs"

ALGORITHM = "KGW"
SAMPLES = 3
RESET_OUTPUTS = True
MATRIX_CONFIG = f"{REPO_DIR}/config/paper_models_l4.json"

print(f"Algorithm: {ALGORITHM}")
print(f"Samples: {SAMPLES}")
print("Models: paper SIRA-Tiny 3B and SIRA-Small 8B configurations")


In [ ]:
# Clone the adapted repository into /content.
from pathlib import Path
import shutil
import subprocess

repo_path = Path(REPO_DIR)
if repo_path.exists():
    shutil.rmtree(repo_path)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True,
)
print("Cloned repository to:", REPO_DIR)


In [ ]:
# Install requirements.
subprocess.run(
    ["pip", "install", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)
subprocess.run(
    ["pip", "install", "--upgrade", "transformers", "accelerate"],
    check=True,
)
print("Requirements installed.")


In [ ]:
# Log in once. Both paper-model checkpoints require accepted Llama access.
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Using the HF_TOKEN Colab Secret.")
else:
    print("No HF_TOKEN found. Gated model runs will be marked skipped.")


In [ ]:
# Start clean and prepare shared environment variables.
import os

if RESET_OUTPUTS and Path(OUTPUT_ROOT).exists():
    shutil.rmtree(OUTPUT_ROOT)
    print("Removed old outputs so every model uses the same fresh data.")

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

models_config_path = f"{OUTPUT_ROOT}/model_runs.json"
env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["ALGORITHM"] = ALGORITHM
env["SAMPLES"] = str(SAMPLES)


In [ ]:
# Run the paper's SIRA-Tiny and SIRA-Small attack models.
subprocess.run(
    [
        "python", "scripts/run_model_matrix_l4.py",
        "--config_path", MATRIX_CONFIG,
        "--repo_dir", REPO_DIR,
        "--output_root", OUTPUT_ROOT,
        "--algorithm", ALGORITHM,
        "--samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

subprocess.run(
    ["python", "scripts/write_environment.py", "--output_path", f"{OUTPUT_ROOT}/environment.json", "--models_config", models_config_path],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

# Run the proposed 2x2 Cognitive Integrity grid-masking comparison.
cognitive_result = subprocess.run(
    [
        "python", "scripts/run_cognitive_integrity_baseline.py",
        "--input_path", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
        "--output_path", f"{OUTPUT_ROOT}/cognitive_integrity/cognitive_integrity_attack.jsonl",
        "--model_name", "meta-llama/Llama-3.2-3B-Instruct",
        "--grid_size", "2",
        "--dtype", "bf16",
        "--max_samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=False,
)
if cognitive_result.returncode != 0:
    print("Cognitive Integrity baseline failed; paper-model evaluation will continue.")


In [ ]:
# Evaluate attack success and semantic preservation against the paper values.
subprocess.run(
    [
        "python", "scripts/evaluate_sira_transfer.py",
        "--generation_model", "facebook/opt-1.3b",
        "--algorithm", ALGORITHM,
        "--watermarked_input", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
        "--models_config", models_config_path,
        "--cognitive_input", f"{OUTPUT_ROOT}/cognitive_integrity/cognitive_integrity_attack.jsonl",
        "--cognitive_grid_size", "2",
        "--output_root", OUTPUT_ROOT,
        "--dtype", "bf16",
        "--max_samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

subprocess.run(
    [
        "python", "scripts/compare_transfer_results.py",
        "--output_root", OUTPUT_ROOT,
        "--algorithm", ALGORITHM,
        "--samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
# Display the final report and paper-style comparison dataframe.
import pandas as pd
from IPython.display import display

report_path = Path(OUTPUT_ROOT) / "final_report.md"
print(report_path.read_text(encoding="utf-8"))

comparison_dataframe = pd.read_csv(f"{OUTPUT_ROOT}/results/paper_style_comparison.csv")
comparison_columns = [
    "method", "paper_method", "model_family", "size_tier", "parameter_size", "run_status",
    "paper_attack_success_rate", "reproduced_attack_success_rate", "difference",
    "semantic_similarity", "logic_trace_verification_rate", "note",
]
display(comparison_dataframe[comparison_columns])

print("\nSaved output files:")
for path in sorted(Path(OUTPUT_ROOT).rglob("*")):
    if path.is_file():
        print(path)


In [ ]:
# Copy results to Google Drive so they survive after the Colab runtime stops.
from google.colab import drive

drive.mount("/content/drive")
drive_output = Path("/content/drive/MyDrive/sira_paper_model_outputs")

if drive_output.exists():
    shutil.rmtree(drive_output)

shutil.copytree(OUTPUT_ROOT, drive_output)
print("Copied results to:", drive_output)


## Stop the Runtime

After the Drive copy finishes, use **Runtime > Disconnect and delete runtime** to release the L4 GPU.
